# Asset Pricing Architectures — CAPM, APT, and Risk-Adjusted Performance

**Market chosen:** USA (NYSE example)  
**Frequency:** Monthly (resampled from daily)  
**Window:** 5 years (60 monthly observations of returns)

This notebook is designed to be **production-ready**: it downloads data, cleans/aligned it, runs regressions, generates diagnostics tables, constructs the required portfolio, computes performance ratios, and produces the CML/SML charts.

## Data specification (what the notebook uses)
- **4 equities** (different sectors): adjusted close prices (monthly)
- **Market portfolio proxy (Rm):** NYSE Composite (default: `^NYA`) adjusted close prices (monthly)
- **Risk-free rate (Rf):** 3-month / 91-day Treasury Bill yield (FRED `TB3MS`, monthly, % annualized)
- **APT macro factors (2):** CPI level (FRED `CPIAUCSL`) and Fed Funds rate (FRED `FEDFUNDS`)

You may replace tickers/series IDs in the configuration cell below if your group decides to use a different market index or macro factor choices.

In [ ]:
import os
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import yfinance as yf
from pandas_datareader import data as pdr

sns.set_theme(style="whitegrid")
pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 50)

OUTPUT_DIR = os.path.join('..', 'outputs')
PROCESSED_DIR = os.path.join('..', 'data', 'processed')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [ ]:
# ---------------------------
# CONFIGURATION (EDIT HERE)
# ---------------------------

# 4 equities from different sectors (example set). Replace if desired.
ASSETS = {
    'AAPL': 'Technology',
    'JPM': 'Financials',
    'XOM': 'Energy',
    'PG': 'Consumer Staples',
}

# Market proxy portfolio (Rm) — choose one:
MARKET_TICKER = '^NYA'   # NYSE Composite
# MARKET_TICKER = '^GSPC'  # S&P 500 (alternative)

# Risk-free: 3-Month Treasury Bill (monthly, annualized %)
RF_SERIES_FRED = 'TB3MS'

# APT macro factors (monthly):
FACTOR_1_FRED = 'CPIAUCSL'   # CPI level
FACTOR_2_FRED = 'FEDFUNDS'   # Fed funds rate (%)

# Analysis window: 5 years of MONTHLY returns (60 points).
N_MONTHS = 60

# Use a buffer so returns have enough history; the final dataset is truncated to the last N_MONTHS.
END_DATE = pd.Timestamp.today().normalize()
START_DATE = (END_DATE - pd.DateOffset(years=6)).strftime('%Y-%m-%d')
END_DATE = END_DATE.strftime('%Y-%m-%d')

WEIGHTS = pd.Series({
    'AAPL': 0.30,
    'JPM': 0.25,
    'XOM': 0.20,
    'PG': 0.25,
})

assert set(WEIGHTS.index) == set(ASSETS.keys())
assert abs(WEIGHTS.sum() - 1.0) < 1e-9

print('Assets:', list(ASSETS.keys()))
print('Market:', MARKET_TICKER)
print('Window (download):', START_DATE, 'to', END_DATE)

In [ ]:
def download_adj_close_yahoo(tickers: list[str], start: str, end: str) -> pd.DataFrame:
    raw = yf.download(
        tickers=tickers,
        start=start,
        end=end,
        interval='1d',
        auto_adjust=False,
        actions=False,
        progress=False,
        group_by='column',
    )
    if raw is None or raw.empty:
        raise RuntimeError('Yahoo Finance download returned empty data. Check internet connectivity and tickers.')

    # Prefer Adj Close; fall back to Close for instruments without adjusted series.
    if 'Adj Close' in raw.columns:
        prices = raw['Adj Close']
    else:
        # MultiIndex case or missing Adj Close
        level0 = raw.columns.get_level_values(0) if isinstance(raw.columns, pd.MultiIndex) else []
        if isinstance(raw.columns, pd.MultiIndex) and 'Adj Close' in level0:
            prices = raw['Adj Close']
        else:
            prices = raw['Close']

    prices = pd.DataFrame(prices).sort_index()
    prices = prices.dropna(how='all')
    # Ensure expected columns exist even if Yahoo returns a Series
    missing = [t for t in tickers if t not in prices.columns]
    if missing:
        raise RuntimeError(f'Missing tickers in Yahoo output: {missing}. Try different tickers or rerun later.')
    return prices

asset_tickers = list(ASSETS.keys())
all_tickers = asset_tickers + [MARKET_TICKER]

prices_daily = download_adj_close_yahoo(all_tickers, START_DATE, END_DATE)
prices_assets_daily = prices_daily[asset_tickers]
prices_mkt_daily = prices_daily[[MARKET_TICKER]].rename(columns={MARKET_TICKER: 'MKT'})

display(prices_assets_daily.tail())
display(prices_mkt_daily.tail())

In [ ]:
# Resample DAILY -> MONTHLY using last available observation (reduces microstructure noise).
prices_assets_m = prices_assets_daily.resample('M').last()
prices_mkt_m = prices_mkt_daily.resample('M').last()

# Log monthly returns: r_t = ln(P_t / P_{t-1})
r_assets = np.log(prices_assets_m / prices_assets_m.shift(1))
r_mkt = np.log(prices_mkt_m['MKT'] / prices_mkt_m['MKT'].shift(1)).rename('MKT')

# Risk-free (Rf): FRED series is annualized percent, monthly frequency
rf_annual_pct = pdr.DataReader(RF_SERIES_FRED, 'fred', START_DATE, END_DATE).resample('M').last().ffill()
rf_annual_rate = (rf_annual_pct[RF_SERIES_FRED] / 100.0).rename('RF_annual')
rf_monthly_simple = ((1 + rf_annual_rate) ** (1/12) - 1).rename('RF')
rf_monthly_log = np.log1p(rf_monthly_simple).rename('RF')

# APT macro factors (FRED)
f1_level = pdr.DataReader(FACTOR_1_FRED, 'fred', START_DATE, END_DATE).resample('M').last().ffill()[FACTOR_1_FRED].rename('CPI')
f2_level = pdr.DataReader(FACTOR_2_FRED, 'fred', START_DATE, END_DATE).resample('M').last().ffill()[FACTOR_2_FRED].rename('FEDFUNDS')

# Convert macro levels -> monthly change/growth forms
F1 = (f1_level.pct_change() * 100.0).rename('Inflation_MoM_%')
F2 = (f2_level.diff()).rename('FedFunds_Δ_pp')

# Align all series and drop missing
df = pd.concat([r_assets, r_mkt, rf_monthly_log, rf_annual_rate, F1, F2], axis=1).dropna()

# Keep only the last N_MONTHS (5 years) of monthly observations
df = df.tail(N_MONTHS)

r_assets = df[asset_tickers]
r_mkt = df['MKT']
rf_monthly_log = df['RF']
rf_annual_rate = df['RF_annual']
factors_raw = df[['Inflation_MoM_%', 'FedFunds_Δ_pp']]

print('Monthly observations (returns):', len(df))
print('Average annualized risk-free rate over window (Rf):', f
)

display(df.head())

# QUESTION ONE — Empirical CAPM Implementation & Regression Diagnostics
We run OLS regressions for each asset using the CAPM excess return structure:

$$ (R_{j,t} - R_{f,t}) = lpha_j + eta_j (R_{m,t} - R_{f,t}) + psilon_{j,t}. $$

Where returns are monthly log returns and $R_{f,t}$ is the monthly log return implied by the 3-month T-bill yield.

In [ ]:
def run_capm(excess_asset: pd.Series, excess_mkt: pd.Series) -> sm.regression.linear_model.RegressionResultsWrapper:
    X = sm.add_constant(excess_mkt.rename('MKT_EXCESS'))
    model = sm.OLS(excess_asset, X).fit()
    return model

excess_assets = r_assets.sub(rf_monthly_log, axis=0)
excess_mkt = (r_mkt - rf_monthly_log).rename('MKT_EXCESS')

capm_models = {t: run_capm(excess_assets[t], excess_mkt) for t in asset_tickers}

capm_summary = []
for t, m in capm_models.items():
    capm_summary.append({
        'Asset': t,
        'Sector': ASSETS[t],
        'Alpha (const)': m.params['const'],
        'Beta': m.params['MKT_EXCESS'],
        'R^2': m.rsquared,
        'SE(Alpha)': m.bse['const'],
        'SE(Beta)': m.bse['MKT_EXCESS'],
        'p(Alpha)': m.pvalues['const'],
        'p(Beta)': m.pvalues['MKT_EXCESS'],
    })

capm_tbl = pd.DataFrame(capm_summary).set_index('Asset')

def classify_beta(beta: float) -> str:
    if abs(beta) < 0.1:
        return 'Risk-Free (≈0)'
    if beta > 1.0:
        return 'Aggressive (β>1)'
    if beta < 1.0:
        return 'Defensive (β<1)'
    return 'Market-like (β≈1)'

capm_tbl['Beta Class'] = capm_tbl['Beta'].apply(classify_beta)
capm_tbl['Alpha Significant (5%)'] = capm_tbl['p(Alpha)'] < 0.05

display(capm_tbl)
capm_tbl.to_csv(os.path.join(OUTPUT_DIR, 'capm_summary.csv'))

# Optional: regression output for each asset
for t, m in capm_models.items():
    print(f